In [5]:
import os.path

from selenium.webdriver.common.by import By
from src.utils.gembase_utils import GembaseUtils
from src.db import DbConnection
from src.utils.web import get_driver
from bs4 import BeautifulSoup, PageElement
from src import external_api

sheet_id = "1ft6DG1S-iOTI6Ps4GtuITvStyHdEZNeM2Y_752ULtjY"
conn = DbConnection()

In [ ]:
conn.close()

In [30]:
def login():
    driver = get_driver('https://app.meettomatch.com/ng2023/site/signin/')
    username = driver.find_element(by=By.NAME, value="data[User][username]")
    password = driver.find_element(by=By.ID, value="password_real")
    form = driver.find_element(by=By.NAME, value="loginForm")
    username.send_keys("pavel.dolezal@gembase.io")
    password.send_keys("0PZd8IEi")
    form.submit()
    sleep(5)
    return driver


In [ ]:
from time import sleep
from selenium.webdriver.remote.webelement import WebElement

driver = login()

company_urls = []

driver.get("https://app.meettomatch.com/ng2023/company/")
sleep(5)
page = 1
while True:
    sleep(5)
    print(f"Page {page}")
    company_names = driver.find_elements(by=By.CLASS_NAME, value="nameDesktop")
    for it in company_names:
        company_a: WebElement = it.find_element(by=By.TAG_NAME, value="a")
        company_url = company_a.get_attribute('href')
        if company_url is not None:
            if company_url not in company_urls:
                company_urls.append(company_url)
    next_button = driver.find_element(by=By.CLASS_NAME, value="nextPage")
    if next_button is not None:
        page += 1
        next_button.click()
    else:
        break

In [ ]:
from src import external_api

values = [['url']]
for it in company_urls:
    values.append([it])
external_api.write_sheet(sheet_id, "company_urls", values)

In [46]:
company_urls = external_api.read_sheet(sheet_id, "company_urls", True)

driver = login()
for url in company_urls:
    try:
        driver.get(url['url'])
        sleep(3)
        data_company = GembaseUtils.compress(driver.page_source)
        detail_button = driver.find_element(by=By.CLASS_NAME, value="company_attendees_people_to_meet_button")
        detail_click = detail_button.get_attribute("onclick")
        people_url = detail_click.replace("goLink('","").replace("', '_self');", "")
        driver.get(people_url)
        sleep(3)
        data_people = GembaseUtils.compress(driver.page_source)
        conn.query("""
        INSERT INTO app_temp_data.temp_nordicgames_scrapper (url, content_company, content_people) VALUES (%s, %s, %s)
        """, [url['url'], data_company, data_people])
        conn.commit()
    except Exception as err:
        conn.rollback()
driver.close()

In [2]:
def parse_url(url: str):
    return url.replace("goLink('","").replace("', '_self');", "").replace("', '_blank');", "")

def parse2(content_company: str, content_people: str):
    soup = BeautifulSoup(content_company, 'html.parser')

    data = {
        'nordic_url': "",
        'company_name': "",
        'company_country': "",
        'company_employees': "",
        'company_url': "",
        'company_profile': "",
        'company_interested': "",
        'contact1_name': "",
        'contact1_url': "",
        'contact1_position': "",
        'contact1_presentation': "",
        'contact1_deal_size': "",
        'contact1_looking_for': "",
        'contact1_industry': "",
        'contact1_platform': "",
        'contact1_invest': "",
    }

    o: PageElement = soup.find(attrs={"class": "attendees_company_details_name"})
    if o is not None:
        data['company_name'] = o.text

    o = soup.find(attrs={"class": "attendees_company_spacing_company_info"})
    if o is not None:
        if " • " in o.text:
            ix = o.text.index(" • ")
            data['company_country'] = o.text[:ix]
            data['company_employees'] = o.text[ix + 3:].replace(" employees", "")
        else:
            data['company_country'] = o.text
    o = soup.find(attrs={"class": "attendees_company_info_container"})
    if o is not None:
        oc = o.find_all("div")
        for it in oc:
            span1 = it.find(attrs={"class": "attendees_contact_info_title"})
            if span1 is not None and span1.text == "Company profile:":
                span2 = it.find(attrs={"class": "people_to_meet_text"})
                data['company_profile'] = span2.text
            if span1 is not None and span1.text == "Is interested in:":
                span2 = it.find(attrs={"class": "people_to_meet_text"})
                data['company_interested'] = span2.text
    o = soup.find(attrs={"class": "company_attendees_advanced_search_buttons"})
    if o is not None:
        h = o.get("onclick")
        if h is not None:
            data['company_url'] = parse_url(h)

    o = soup.find(attrs={"class": "company_attendees_people_to_meet_button"})
    if o is not None:
        h = o.get("onclick")
        if h is not None:
            data['contact1_url'] = parse_url(h)

    soup = BeautifulSoup(content_people, 'html.parser')

    o = soup.find(attrs={"class": "attendees_company_details_name"})
    if o is not None:
        data['contact1_name'] = o.text

    o = soup.find(attrs={"class": "attendees_company_spacing_company_info"})
    if o is not None:
        o1 = o.find(attrs={"class": "people_to_meet_text"})
        if o1 is not None:
            data['contact1_position'] = o1.text

    o = soup.find_all(attrs={"class": "attendees_contact_info_title"})
    for it in o:
        if it.text == "Online presentation, profile or pitch deck":
            p = it.find_parent()
            a = p.find("a")
            if a is not None:
                h = a.get("href")
                if h is not None:
                    data['contact1_presentation'] = h
        elif it.text == "Preferred deal size":
            p = it.find_next_sibling("div")
            spans = p.find_all(attrs={"class": "attendees_company_looking_for_buttons_text"})
            arr = []
            for span in spans:
                arr.append(span.text)
            data['contact1_deal_size'] = ','.join(arr)
        elif it.text == "Looking for investment type":
            p = it.find_next_sibling("div")
            spans = p.find_all(attrs={"class": "attendees_company_looking_for_buttons_text"})
            arr = []
            for span in spans:
                arr.append(span.text)
            data['contact1_invest'] = ','.join(arr)
        elif it.text == "Industry":
            p = it.find_next_sibling("div")
            spans = p.find_all(attrs={"class": "attendees_company_looking_for_buttons_text"})
            arr = []
            for span in spans:
                arr.append(span.text)
            data['contact1_industry'] = ','.join(arr)
        elif it.text == "Platform":
            p = it.find_next_sibling("div")
            spans = p.find_all(attrs={"class": "attendees_company_looking_for_buttons_text"})
            arr = []
            for span in spans:
                arr.append(span.text)
            data['contact1_platform'] = ','.join(arr)
        elif it.text == "Looking for":
            p = it.find_next_sibling("div")
            spans = p.find_all(attrs={"class": "attendees_company_looking_for_buttons_text"})
            arr = []
            for span in spans:
                arr.append(span.text)
            data['contact1_looking_for'] = ','.join(arr)

    return data



In [12]:
rows = conn.select_all("""
SELECT n.id, n.dev_id, n.url, n.content_company, n.content_people
FROM app_temp_data.temp_nordicgames_scrapper n
""")

data_arr = []
for row in rows:
    try:
        data = parse2(GembaseUtils.decompress(row['content_company']), GembaseUtils.decompress(row['content_people']))
        for k in data:
            data[k] = data[k].removeprefix("\n").removesuffix("\n")
        data['nordic_url'] = row['url']
        data['id'] = row['id']
        data['dev_id'] = row['dev_id']
        data['dev_title'] = ""
        data['dev_url'] = ""
        data['dev_apps'] = ""
        if data['dev_id'] is not None and data['dev_id'] != 'N/A':
            row_dev = conn.select_one_or_none("""
            SELECT d.name, d.link FROM google_play_scraper.gps_devs d WHERE d.store_id = %s
            """, [data['dev_id']])
            if row_dev is not None:
                data['dev_title'] = row_dev['name']
                data['dev_url'] = row_dev['link']
                row_apps = conn.select_all("""
                SELECT g.google_id, g.title, g.real_installs, g.url FROM google_play_scraper.gps_games g WHERE g.developer_id = %s
                """, [data['dev_id']])
                arr = []
                for row_app in row_apps:
                    arr.append(f"{row_app['title']} - {row_app['real_installs']} - {row_app['url']}")
                data['dev_apps'] = '\n'.join(arr)
        data_arr.append(data)
    except Exception as err:
        print(err)

In [11]:
from src.server.models.google_play_scrapper_model import scrap_developer, search_google_play_developers

for it in data_arr:
    if it['dev_id'] is not None:
        continue
    d = search_google_play_developers(it['company_name'], True)
    if len(d) > 0:
        print(f"Developer {it['company_name']}: {d[0]}")
        it['dev_id'] = d[0]['store_id']
        row = conn.select_one_or_none("""
        SELECT * FROM google_play_scraper.gps_devs d WHERE d.store_id = %s
        """, [it['dev_id']])
        if row is None:
            scrap_developer(conn, it['dev_id'], save_to_db=True)
            row2 = conn.select_one_or_none("""
            SELECT * FROM google_play_scraper.gps_devs d WHERE d.store_id = %s
            """, [it['dev_id']])
            if row2 is not None:
                conn.query("""
                UPDATE app_temp_data.temp_nordicgames_scrapper n SET n.dev_id = %s
                WHERE n.id = %s
                """, [it['dev_id'], it['id']])
        else:
            conn.query("""
            UPDATE app_temp_data.temp_nordicgames_scrapper n SET n.dev_id = %s
            WHERE n.id = %s
            """, [it['dev_id'], it['id']])
    else:
        it['dev_id'] = 'N/A'
        conn.query("""
        UPDATE app_temp_data.temp_nordicgames_scrapper n SET n.dev_id = 'N/A'
        WHERE n.id = %s
        """, [it['id']])
    conn.commit()


Developer Kwalee: {'store_id': '7085784401169971431', 'name': 'Kwalee'}
Developer Bloom & Gloom Games: {'store_id': 'TAMK+Tietojenk%C3%A4sittelyn+koulutus', 'name': 'TAMK Tietojenkäsittelyn koulutus'}
Developer Green man Gaming: {'store_id': 'Green+Man', 'name': 'Green Man'}
Developer Twigames: {'store_id': '8984012638712995182', 'name': 'Twigames Inc.'}
Developer Coffee Stain Malmö: {'store_id': '6719114643248126714', 'name': 'Coffee Stain Publishing'}
Developer Gameforge 4D GmbH: {'store_id': 'Gameforge+4D+GmbH', 'name': 'Gameforge 4D GmbH'}
Developer GAMES ACADEMY: {'store_id': 'Girl+Games+Academy', 'name': 'Girl Games Academy'}
Developer Torpor Games: {'store_id': '8036152237901494618', 'name': 'Torpor Games'}
Developer Pixel Federation: {'store_id': '7566304266577611559', 'name': 'Pixel Federation Games'}
Developer Striked GmbH: {'store_id': '5099160657148662997', 'name': 'Edkon Games GmbH'}
Developer Definite Studios: {'store_id': 'Prosults+Studio', 'name': 'Prosults Studio'}
Dev

In [13]:
values = []
if len(data_arr) > 0:
    values.append([])
    for k in data_arr[0]:
        values[0].append(k)
    for data in data_arr:
        arr = [""] * len(values[0])
        values.append(arr)
        for k in data:
            ix = values[0].index(k)
            arr[ix] = data[k]

external_api.write_sheet(sheet_id, "data", values)

{'spreadsheetId': '1ft6DG1S-iOTI6Ps4GtuITvStyHdEZNeM2Y_752ULtjY',
 'updatedRange': 'data!A1:U418',
 'updatedRows': 418,
 'updatedColumns': 21,
 'updatedCells': 8778}

In [ ]:
conn.close()